- 1. Call NewsAPI and get articles
- 2. Save the response as a JSON file into volume
- 3. Use Auto Loader to read from the Volume into a delta table

In [0]:
import requests
import json
API_KEY = 'a89ebff894334c3e8c78972672c714f0'
URL = f'https://newsapi.org/v2/everything?q=india&language=en&apiKey={API_KEY}'
response = requests.get(URL)
data = response.json()

In [0]:
print(f"Status: {data['status']}")
print(f"Total results: {data['totalResults']}")
print(f"Articles fetched: {len(data['articles'])}")

- Convert the response dictionary to JSON and write it into the file with current timestamp
- Close the file automatically

In [0]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
filename = f"articles_{timestamp}.json"
filepath = f"/Volumes/news_pipeline/bronze/raw_json_files/{filename}"
with open(filepath, 'w') as f:
  for article in data['articles']:
    f.write(json.dumps(article) + '\n')
print(f"Saved {len(data['articles'])} articles to {filepath}")

In [0]:
dbutils.fs.ls("/Volumes/news_pipeline/bronze/raw_json_files")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("author", StringType(), True),
    StructField("content", StringType(), True),
    StructField("description", StringType(), True),
    StructField("publishedAt", StringType(), True),
    StructField("source",StructType([
        StructField("id", StringType(), True),
        StructField("name", StringType(), True)   
    ])),
    StructField("title", StringType(), True),
    StructField("url", StringType(), True),
    StructField("urlToImage", StringType(), True)
])

In [0]:
spark.readStream.format("cloudFiles") \
    .schema(schema) \
    .option("cloudFiles.format","json") \
    .option("cloudFiles.schemaLocation", "/Volumes/news_pipeline/bronze/raw_json_files/_schema") \
    .load("/Volumes/news_pipeline/bronze/raw_json_files") \
    .writeStream.option("checkpointLocation","/Volumes/news_pipeline/bronze/raw_json_files/_checkpoints") \
    .trigger(availableNow=True) \
    .toTable("news_pipeline.bronze.raw_articles")


In [0]:
df = spark.table("news_pipeline.bronze.raw_articles")
df.printSchema()